In [ ]:
# Treino CNN - ResNet50, EfficientNet-B0, ConvNeXt-Tiny e DenseNet121 (P2 · Tema F)
# Fine-tuning de quatro CNN pré-treinadas (ImageNet) no FieldVitis, com o `splits.json` do repositório (3 seeds).

In [1]:

# --- Configuração dos dados locais e dos splits ---
import json
from pathlib import Path
import pandas as pd

# Encontrar automaticamente a raiz do repositório.
# Funciona se o notebook for executado na raiz ou dentro de notebooks/.
cwd = Path.cwd().resolve()

REPO_DIR = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "data" / "fieldvitis").is_dir()
        and (p / "data" / "splits.json").is_file()
    ),
    None
)

if REPO_DIR is None:
    raise FileNotFoundError(
        "Não foi possível encontrar a raiz do repositório."
    )

# Caminhos locais
DATA_DIR = REPO_DIR / "data" / "fieldvitis"

QUALITY_DIR = REPO_DIR / "data" / "qualidade"
RAW_MANIFEST_FILE = QUALITY_DIR / "manifest_raw.csv"
REMOVIDAS_FILE = QUALITY_DIR / "removidas.csv"

HOLDOUT_FILE = REPO_DIR / "data" / "holdout.csv"
MANIFEST_FILE = REPO_DIR / "data" / "manifest_clean.csv"
SPLITS_FILE = REPO_DIR / "data" / "splits.json"

# Confirmar que todos os ficheiros necessários existem
required_paths = [
    DATA_DIR,
    RAW_MANIFEST_FILE,
    REMOVIDAS_FILE,
    HOLDOUT_FILE,
    MANIFEST_FILE,
    SPLITS_FILE
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Não encontrado: {path}")

# Carregar os manifestos
manifest_raw = pd.read_csv(RAW_MANIFEST_FILE)
removidas = pd.read_csv(REMOVIDAS_FILE)
manifest = pd.read_csv(MANIFEST_FILE)

# Carregar os splits
with SPLITS_FILE.open(encoding="utf-8") as f:
    sj = json.load(f)

CLASSES = sj["classes"]
SEEDS = [0, 1, 2]

# Identificar o Holdout através das duas fontes
holdout_json = set(sj["holdout"])
holdout_csv = set(
    pd.read_csv(HOLDOUT_FILE)["path"].astype(str)
)

assert holdout_json == holdout_csv, (
    "Os caminhos do Holdout em splits.json e holdout.csv não coincidem."
)

HOLDOUT_PATHS = holdout_json

# Definir o conjunto elegível: manifesto limpo menos Holdout
clean_paths = set(manifest["path"].astype(str))

assert HOLDOUT_PATHS <= clean_paths, (
    "Há caminhos do Holdout que não constam do manifesto limpo."
)

pool_paths = clean_paths - HOLDOUT_PATHS

# Informação geral
print("Raiz do projeto:", REPO_DIR)
print("Dataset:", DATA_DIR)
print("Manifesto bruto:", len(manifest_raw))
print("Imagens removidas:", len(removidas))
print("Imagens no manifesto limpo:", len(clean_paths))
print("Imagens reservadas (Holdout):", len(HOLDOUT_PATHS))
print("Imagens elegíveis (sem Holdout):", len(pool_paths))

# Verificar os três conjuntos de cada seed
for seed in SEEDS:
    sp = sj["seeds"][str(seed)]

    split_sets = {
        k: set(sp[k])
        for k in ("train", "val", "test")
    }

    assert not (split_sets["train"] & split_sets["val"])
    assert not (split_sets["train"] & split_sets["test"])
    assert not (split_sets["val"] & split_sets["test"])

    assigned = set.union(*split_sets.values())

    assert assigned == pool_paths, (
        f"A seed {seed} não corresponde ao conjunto não-Holdout."
    )

    assert not (assigned & HOLDOUT_PATHS), (
        f"Fuga do Holdout detetada na seed {seed}!"
    )

    faltam = [
        p for p in assigned
        if not (DATA_DIR / p).is_file()
    ]

    print(
        f"Seed {seed}:",
        {k: len(v) for k, v in split_sets.items()},
        "| ficheiros em falta:", len(faltam),
        "| Holdout incluído: 0"
    )

    assert not faltam, (
        f"Há imagens dos splits que não existem em {DATA_DIR}."
    )

print("\nVerificação concluída: apenas imagens não-Holdout.")

Raiz do projeto: /home/xy/Projects/DeepLearning_notebooks/DL_P2_TP2_G2-F1
Dataset: /home/xy/Projects/DeepLearning_notebooks/DL_P2_TP2_G2-F1/data/fieldvitis
Manifesto bruto: 555
Imagens removidas: 0
Imagens no manifesto limpo: 555
Imagens reservadas (Holdout): 50
Imagens elegíveis (sem Holdout): 505
Seed 0: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0
Seed 1: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0
Seed 2: {'train': 352, 'val': 76, 'test': 77} | ficheiros em falta: 0 | Holdout incluído: 0

Verificação concluída: apenas imagens não-Holdout.


In [2]:
# Configuração e Carregamento das imagens
import time, gc
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import f1_score, recall_score

IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, LR_HEAD = 5, 1e-3     # fase 1: só a cabeça (base congelada)
EPOCHS_FT, LR_FT = 25, 1e-5        # fase 2: fine-tuning da rede toda
PATIENCE = 6                       # early stopping no Macro-F1 de validação
SEEDS = [0, 1, 2]
RUNS_DIR = REPO_DIR / "runs" / "P2_runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| GPU:", tf.config.list_physical_devices("GPU"))

# Todas as imagens carregadas uma vez: RGB, orientação EXIF corrigida, 224x224
all_paths = sorted({p for s in sj["seeds"].values() for k in ("train", "val", "test") for p in s[k]})
cls_idx = {c: i for i, c in enumerate(CLASSES)}
IMGS, LABELS = {}, {}
for p in all_paths:
    im = ImageOps.exif_transpose(Image.open(DATA_DIR / p)).convert("RGB")
    IMGS[p] = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    LABELS[p] = cls_idx[Path(p).parts[0]]
print("Imagens carregadas:", len(IMGS), "| por classe:", np.bincount(list(LABELS.values())))

holdout_carregadas = len(set(IMGS) & HOLDOUT_PATHS)

print(f"Imagens Holdout carregadas: {holdout_carregadas}")

assert holdout_carregadas == 0, (
    "Foram carregadas imagens do Holdout!"
)

I0000 00:00:1794956878.426623   71098 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794956878.455487   71098 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794956879.205220   71098 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Keras 3.15.1 | backend tensorflow | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Imagens carregadas: 505 | por classe: [ 35 182 144 144]
Imagens Holdout carregadas: 0


In [3]:
# Cada rede espera os píxeis num formato diferente (o mesmo usado no treino do ImageNet)
PREPROCESS = {
    "resnet50": keras.applications.resnet50.preprocess_input,           # converte para BGR e subtrai a média do ImageNet
    "efficientnetb0": keras.applications.efficientnet.preprocess_input,  # não altera: o modelo normaliza por dentro
    "convnext_tiny": keras.applications.convnext.preprocess_input,       # não altera: o modelo normaliza por dentro
    "densenet121": keras.applications.densenet.preprocess_input,         # escala para [0,1] e normaliza com média/desvio do ImageNet
}

def preparar_dados(seed, nome):
    """Devolve treino/val/teste da seed indicada, já pré-processados para o modelo."""
    sp = sj["seeds"][str(seed)]
    dados = {}
    for split in ["train", "val", "test"]:
        x = np.stack([IMGS[p] for p in sp[split]]).astype("float32")
        y = np.array([LABELS[p] for p in sp[split]])
        dados[split] = (PREPROCESS[nome](x), y, sp[split])
    return dados

In [4]:
def criar_modelo(nome):
    """Base pré-treinada no ImageNet + augmentation + camada final de 4 classes."""
    if nome == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "efficientnetb0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    elif nome == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights="imagenet",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    else:
        base = keras.applications.DenseNet121(include_top=False, weights="imagenet",
                                              input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])  # só atua durante o treino

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)   # BatchNorm fica em modo inferência, mesmo no fine-tuning
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

In [5]:
# Treinar e Guardar
def guardar_previsoes(nome, seed, split, caminhos, y, probs):
    """CSV para o avaliador: uma linha por imagem, com a classe verdadeira, a prevista e as 4 probabilidades."""
    df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
    df.insert(0, "y_pred", [CLASSES[i] for i in probs.argmax(axis=1)])
    df.insert(0, "y_true", [CLASSES[i] for i in y])
    df.insert(0, "path", caminhos)
    df.insert(0, "split", split)
    df.insert(0, "seed", seed)
    df.insert(0, "modelo", nome)
    df.to_csv(RUNS_DIR / f"preds_{nome}_seed{seed}_{split}.csv", index=False)

class EarlyStoppingMacroF1(keras.callbacks.Callback):
    """Early stopping no Macro-F1 de validação (maximizar); restaura os melhores pesos no fim."""
    def __init__(self, x_val, y_val, patience):
        super().__init__()
        self.x_val, self.y_val, self.patience = x_val, y_val, patience

    def on_train_begin(self, logs=None):
        self.best, self.wait, self.best_weights = -1.0, 0, None

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.x_val, batch_size=BATCH, verbose=0).argmax(axis=1)
        f1 = f1_score(self.y_val, pred, average="macro")
        print(f"    val_macro_f1 = {f1:.4f} (melhor: {max(self.best, f1):.4f})")
        if f1 > self.best:
            self.best, self.wait = f1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

def treinar(nome, seed):
    keras.utils.set_random_seed(seed)   # reprodutibilidade
    dados = preparar_dados(seed, nome)
    x_tr, y_tr, _ = dados["train"]
    x_va, y_va, _ = dados["val"]

    # Pesos por classe: o black rot (poucas imagens) pesa mais na loss
    contagens = np.bincount(y_tr, minlength=len(CLASSES))
    class_weight = {i: len(y_tr) / (len(CLASSES) * n) for i, n in enumerate(contagens)}

    model, base = criar_modelo(nome)

    # Fase 1: treina só a camada final (base congelada)
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_HEAD,
              batch_size=BATCH, class_weight=class_weight, verbose=2)

    # Fase 2: fine-tuning de toda a rede, com learning rate baixo
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    early = EarlyStoppingMacroF1(x_va, y_va, PATIENCE)
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_FT,
              batch_size=BATCH, class_weight=class_weight, callbacks=[early], verbose=2)

    # Guardar o modelo (só no Drive) e as previsões de val e teste (para o avaliador)
    model.save(RUNS_DIR / f"{nome}_seed{seed}.keras")
    for split in ["val", "test"]:
        x, y, caminhos = dados[split]
        guardar_previsoes(nome, seed, split, caminhos, y, model.predict(x, verbose=0))

    _, val_acc = model.evaluate(x_va, y_va, verbose=0)
    print(f">>> {nome} seed {seed}: accuracy de validação = {val_acc:.3f}")

In [6]:
# ResNet50
for seed in SEEDS:
    print(f"\n===== ResNet50 | seed {seed} =====")
    treinar("resnet50", seed)
    keras.backend.clear_session()   # liberta a memória da GPU entre corridas


===== ResNet50 | seed 0 =====


I0000 00:00:1794957187.045753   71098 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6094 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 18s 0us/step
Epoch 1/5


I0000 00:00:1794957211.794364   74164 cuda_dnn.cc:442] Loaded cuDNN version 92801


11/11 - 8s - 736ms/step - accuracy: 0.3750 - loss: 1.5145 - val_accuracy: 0.6974 - val_loss: 0.8869
Epoch 2/5
11/11 - 1s - 108ms/step - accuracy: 0.6364 - loss: 0.9629 - val_accuracy: 0.8684 - val_loss: 0.5000
Epoch 3/5
11/11 - 1s - 106ms/step - accuracy: 0.7812 - loss: 0.6001 - val_accuracy: 0.9079 - val_loss: 0.3415
Epoch 4/5
11/11 - 1s - 105ms/step - accuracy: 0.8239 - loss: 0.5081 - val_accuracy: 0.9342 - val_loss: 0.2611
Epoch 5/5
11/11 - 1s - 109ms/step - accuracy: 0.8665 - loss: 0.3763 - val_accuracy: 0.9474 - val_loss: 0.2185
Epoch 1/25
    val_macro_f1 = 0.9013 (melhor: 0.9013)
11/11 - 17s - 2s/step - accuracy: 0.8864 - loss: 0.3727 - val_accuracy: 0.9342 - val_loss: 0.2330
Epoch 2/25
    val_macro_f1 = 0.9563 (melhor: 0.9563)
11/11 - 3s - 262ms/step - accuracy: 0.9148 - loss: 0.2651 - val_accuracy: 0.9737 - val_loss: 0.1338
Epoch 3/25
    val_macro_f1 = 0.9160 (melhor: 0.9563)
11/11 - 3s - 245ms/step - accuracy: 0.9602 - loss: 0.1486 - val_accuracy: 0.9605 - val_loss: 0.1750


In [7]:
# EfficientNet-B0
for seed in SEEDS:
    print(f"\n===== EfficientNet-B0 | seed {seed} =====")
    treinar("efficientnetb0", seed)
    keras.backend.clear_session()


===== EfficientNet-B0 | seed 0 =====
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/5
11/11 - 5s - 486ms/step - accuracy: 0.4688 - loss: 1.2107 - val_accuracy: 0.6316 - val_loss: 1.0175
Epoch 2/5
11/11 - 1s - 64ms/step - accuracy: 0.7017 - loss: 0.9198 - val_accuracy: 0.7105 - val_loss: 0.8300
Epoch 3/5
11/11 - 1s - 63ms/step - accuracy: 0.7585 - loss: 0.7661 - val_accuracy: 0.8026 - val_loss: 0.7099
Epoch 4/5
11/11 - 1s - 65ms/step - accuracy: 0.8011 - loss: 0.7021 - val_accuracy: 0.8289 - val_loss: 0.6287
Epoch 5/5
11/11 - 1s - 63ms/step - accuracy: 0.8438 - loss: 0.6088 - val_accuracy: 0.8421 - val_loss: 0.5811
Epoch 1/25


E0000 00:00:1794957456.072962   71098 meta_optimizer.cc:996] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


    val_macro_f1 = 0.8513 (melhor: 0.8513)
11/11 - 13s - 1s/step - accuracy: 0.8608 - loss: 0.5385 - val_accuracy: 0.8947 - val_loss: 0.4979
Epoch 2/25
    val_macro_f1 = 0.8548 (melhor: 0.8548)
11/11 - 2s - 188ms/step - accuracy: 0.8920 - loss: 0.5410 - val_accuracy: 0.9079 - val_loss: 0.4535
Epoch 3/25
    val_macro_f1 = 0.8678 (melhor: 0.8678)
11/11 - 2s - 184ms/step - accuracy: 0.8693 - loss: 0.4735 - val_accuracy: 0.9079 - val_loss: 0.4314
Epoch 4/25
    val_macro_f1 = 0.8848 (melhor: 0.8848)
11/11 - 2s - 180ms/step - accuracy: 0.9119 - loss: 0.4056 - val_accuracy: 0.9211 - val_loss: 0.3837
Epoch 5/25
    val_macro_f1 = 0.8825 (melhor: 0.8848)
11/11 - 2s - 178ms/step - accuracy: 0.9176 - loss: 0.3811 - val_accuracy: 0.9342 - val_loss: 0.3294
Epoch 6/25
    val_macro_f1 = 0.8952 (melhor: 0.8952)
11/11 - 2s - 183ms/step - accuracy: 0.9261 - loss: 0.3483 - val_accuracy: 0.9342 - val_loss: 0.3134
Epoch 7/25
    val_macro_f1 = 0.8952 (melhor: 0.8952)
11/11 - 2s - 185ms/step - accuracy:

E0000 00:00:1794957559.706533   71098 meta_optimizer.cc:996] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


    val_macro_f1 = 0.7252 (melhor: 0.7252)
11/11 - 13s - 1s/step - accuracy: 0.8381 - loss: 0.6251 - val_accuracy: 0.7895 - val_loss: 0.5931
Epoch 2/25
    val_macro_f1 = 0.8055 (melhor: 0.8055)
11/11 - 2s - 185ms/step - accuracy: 0.8750 - loss: 0.4819 - val_accuracy: 0.8684 - val_loss: 0.5114
Epoch 3/25
    val_macro_f1 = 0.8209 (melhor: 0.8209)
11/11 - 2s - 181ms/step - accuracy: 0.8949 - loss: 0.4693 - val_accuracy: 0.8816 - val_loss: 0.4723
Epoch 4/25
    val_macro_f1 = 0.8653 (melhor: 0.8653)
11/11 - 2s - 181ms/step - accuracy: 0.8949 - loss: 0.4667 - val_accuracy: 0.9079 - val_loss: 0.4630
Epoch 5/25
    val_macro_f1 = 0.8653 (melhor: 0.8653)
11/11 - 2s - 179ms/step - accuracy: 0.8949 - loss: 0.4438 - val_accuracy: 0.9079 - val_loss: 0.4317
Epoch 6/25
    val_macro_f1 = 0.8661 (melhor: 0.8661)
11/11 - 2s - 190ms/step - accuracy: 0.8977 - loss: 0.4444 - val_accuracy: 0.9079 - val_loss: 0.3979
Epoch 7/25
    val_macro_f1 = 0.8661 (melhor: 0.8661)
11/11 - 2s - 183ms/step - accuracy:

In [8]:
# ConvNeXt-Tiny
for seed in SEEDS:
    print(f"\n===== ConvNeXt-Tiny | seed {seed} =====")
    treinar("convnext_tiny", seed)
    keras.backend.clear_session()



===== ConvNeXt-Tiny | seed 0 =====
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step
Epoch 1/5


I0000 00:00:1794957633.373870   74163 service.cc:154] XLA service 0x71f2a018afc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794957633.373887   74163 service.cc:170]   StreamExecutor [0]: NVIDIA GeForce RTX 4070 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0[595.99.2]; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.28.1)
I0000 00:00:1794957633.904029   74163 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1794957635.694032   94783 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
W0000 00:00:1794957640.823006   96446 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


11/11 - 10s - 948ms/step - accuracy: 0.3466 - loss: 1.8271 - val_accuracy: 0.4868 - val_loss: 1.1370
Epoch 2/5
11/11 - 2s - 163ms/step - accuracy: 0.5483 - loss: 1.2674 - val_accuracy: 0.5921 - val_loss: 0.9184
Epoch 3/5
11/11 - 2s - 164ms/step - accuracy: 0.6080 - loss: 1.0894 - val_accuracy: 0.6316 - val_loss: 0.7893
Epoch 4/5
11/11 - 2s - 167ms/step - accuracy: 0.6420 - loss: 0.8932 - val_accuracy: 0.7895 - val_loss: 0.6165
Epoch 5/5
11/11 - 2s - 168ms/step - accuracy: 0.7727 - loss: 0.6324 - val_accuracy: 0.8684 - val_loss: 0.4862
Epoch 1/25


E0000 00:00:1794957660.105089   74163 cuda_executor.cc:1365] [0] Failed to allocate device memory of 11.90GiB (12781223936 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:230
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:487

E0000 00:00:1794957660.105126   74163 cuda_executor.cc:1365] [0] Failed to allocate device memory of 10.71GiB (11503100928 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:230
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:487

E0000 00:00:1794957660.105145   74163 cuda_executor.cc:1365] [0] Failed to allocate device memory of 9.64GiB (10352790528 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_

    val_macro_f1 = 0.7942 (melhor: 0.7942)
11/11 - 26s - 2s/step - accuracy: 0.7670 - loss: 0.7159 - val_accuracy: 0.8553 - val_loss: 0.4443
Epoch 2/25
    val_macro_f1 = 0.8368 (melhor: 0.8368)
11/11 - 6s - 534ms/step - accuracy: 0.7869 - loss: 0.5530 - val_accuracy: 0.8947 - val_loss: 0.3917
Epoch 3/25
    val_macro_f1 = 0.8470 (melhor: 0.8470)
11/11 - 6s - 519ms/step - accuracy: 0.8438 - loss: 0.5179 - val_accuracy: 0.9079 - val_loss: 0.3576
Epoch 4/25
    val_macro_f1 = 0.8459 (melhor: 0.8470)
11/11 - 6s - 524ms/step - accuracy: 0.8182 - loss: 0.5871 - val_accuracy: 0.9079 - val_loss: 0.3321
Epoch 5/25
    val_macro_f1 = 0.8923 (melhor: 0.8923)
11/11 - 6s - 523ms/step - accuracy: 0.8778 - loss: 0.3939 - val_accuracy: 0.9342 - val_loss: 0.3085
Epoch 6/25
    val_macro_f1 = 0.9172 (melhor: 0.9172)
11/11 - 6s - 520ms/step - accuracy: 0.8523 - loss: 0.4518 - val_accuracy: 0.9474 - val_loss: 0.2778
Epoch 7/25
    val_macro_f1 = 0.9172 (melhor: 0.9172)
11/11 - 6s - 520ms/step - accuracy:

W0000 00:00:1794957740.310758  104787 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


>>> convnext_tiny seed 0: accuracy de validação = 0.947

===== ConvNeXt-Tiny | seed 1 =====
Epoch 1/5
11/11 - 9s - 822ms/step - accuracy: 0.4006 - loss: 1.5727 - val_accuracy: 0.5395 - val_loss: 1.0405
Epoch 2/5
11/11 - 2s - 164ms/step - accuracy: 0.6023 - loss: 1.0839 - val_accuracy: 0.6579 - val_loss: 0.7524
Epoch 3/5
11/11 - 2s - 161ms/step - accuracy: 0.6790 - loss: 0.9483 - val_accuracy: 0.7368 - val_loss: 0.6324
Epoch 4/5
11/11 - 2s - 165ms/step - accuracy: 0.6932 - loss: 0.7878 - val_accuracy: 0.8421 - val_loss: 0.5102
Epoch 5/5
11/11 - 2s - 166ms/step - accuracy: 0.7699 - loss: 0.6333 - val_accuracy: 0.8684 - val_loss: 0.3994
Epoch 1/25
    val_macro_f1 = 0.8729 (melhor: 0.8729)
11/11 - 24s - 2s/step - accuracy: 0.8239 - loss: 0.5586 - val_accuracy: 0.9079 - val_loss: 0.3538
Epoch 2/25
    val_macro_f1 = 0.8830 (melhor: 0.8830)
11/11 - 6s - 521ms/step - accuracy: 0.8693 - loss: 0.5258 - val_accuracy: 0.9211 - val_loss: 0.3332
Epoch 3/25
    val_macro_f1 = 0.8729 (melhor: 0.8830

In [9]:
# DenseNet121
for seed in SEEDS:
    print(f"\n===== DenseNet121 | seed {seed} =====")
    treinar("densenet121", seed)
    keras.backend.clear_session()



===== DenseNet121 | seed 0 =====
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/5
11/11 - 9s - 788ms/step - accuracy: 0.3182 - loss: 1.5762 - val_accuracy: 0.5395 - val_loss: 1.1810
Epoch 2/5
11/11 - 1s - 111ms/step - accuracy: 0.4631 - loss: 1.3132 - val_accuracy: 0.7105 - val_loss: 0.9815
Epoch 3/5
11/11 - 1s - 110ms/step - accuracy: 0.5938 - loss: 1.0403 - val_accuracy: 0.8158 - val_loss: 0.8018
Epoch 4/5
11/11 - 1s - 109ms/step - accuracy: 0.6477 - loss: 0.9386 - val_accuracy: 0.8158 - val_loss: 0.6954
Epoch 5/5
11/11 - 1s - 108ms/step - accuracy: 0.6989 - loss: 0.7987 - val_accuracy: 0.8421 - val_loss: 0.6264
Epoch 1/25
    val_macro_f1 = 0.7618 (melhor: 0.7618)
11/11 - 23s - 2s/step - accuracy: 0.7784 - loss: 0.7518 - val_accuracy: 0.8421 - val_loss: 0.4917
Epoch 2/25
    val_macro_f1 = 0.8699 (melhor: 0.8699)
11/11 - 3s - 246ms/step - accuracy: 0.7727 - loss: 0.6463 - val_accuracy: 0.9211 - val_loss: 0.3242
Epoch 3/25
    val_macro_f1 = 0.9060 (melhor: 0.9060)
11/11

In [10]:
import glob
for f in sorted(glob.glob("/content/drive/MyDrive/P2_runs/preds_*_val.csv")):
    df = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s} accuracy val = {(df.y_true == df.y_pred).mean():.3f}")

In [11]:
# Métricas por seed e média das 3 seeds (conjunto de teste)
from sklearn.metrics import accuracy_score
linhas = []
for f in sorted(glob.glob(str(RUNS_DIR / "preds_*_test.csv"))):
    df = pd.read_csv(f)
    linhas.append({"modelo": df.modelo[0], "seed": df.seed[0],
                   "accuracy": accuracy_score(df.y_true, df.y_pred),
                   "macro_f1": f1_score(df.y_true, df.y_pred, average="macro"),
                   "macro_recall": recall_score(df.y_true, df.y_pred, average="macro")})
por_seed = pd.DataFrame(linhas)
print("Métricas por seed:")
print(por_seed.round(4).to_string(index=False))
print("\nMédia das 3 seeds por modelo:")
print(por_seed.groupby("modelo")[["accuracy", "macro_f1", "macro_recall"]].mean().round(4))


Métricas por seed:
        modelo  seed  accuracy  macro_f1  macro_recall
 convnext_tiny     0    0.9091    0.8733        0.8902
 convnext_tiny     1    0.8831    0.8400        0.8998
 convnext_tiny     2    0.9481    0.9125        0.9545
   densenet121     0    0.9610    0.9334        0.9680
   densenet121     1    0.9481    0.9125        0.9545
   densenet121     2    0.9481    0.9443        0.9609
efficientnetb0     0    0.9481    0.9159        0.9263
efficientnetb0     1    0.8831    0.8419        0.8998
efficientnetb0     2    0.8701    0.8338        0.8948
      resnet50     0    0.9740    0.9614        0.9470
      resnet50     1    0.9740    0.9643        0.9773
      resnet50     2    0.9870    0.9750        0.9886

Média das 3 seeds por modelo:
                accuracy  macro_f1  macro_recall
modelo                                          
convnext_tiny     0.9134    0.8753        0.9148
densenet121       0.9524    0.9301        0.9611
efficientnetb0    0.9004    0.8639     

In [12]:
# Custo computacional: parâmetros e latência de inferência (modelos da seed 0, imagens de validação)
custo = []
for nome in ["resnet50", "efficientnetb0", "convnext_tiny", "densenet121"]:
    model = keras.models.load_model(RUNS_DIR / f"{nome}_seed0.keras")
    x, _, _ = preparar_dados(0, nome)["val"]

    model.predict(x[:BATCH], verbose=0)                 # aquecimento
    t0 = time.perf_counter()
    model.predict(x, batch_size=BATCH, verbose=0)
    ms_lote = (time.perf_counter() - t0) / len(x) * 1000

    model(x[:1], training=False)                        # aquecimento
    t0 = time.perf_counter()
    for i in range(20):
        model(x[i:i+1], training=False)
    ms_uma = (time.perf_counter() - t0) / 20 * 1000

    custo.append(dict(modelo=nome,
                      parametros_milhoes=round(model.count_params() / 1e6, 2),
                      ms_por_imagem_em_lote=round(ms_lote, 1),
                      ms_por_imagem_isolada=round(ms_uma, 1)))
    del model; keras.backend.clear_session(); gc.collect()

display(pd.DataFrame(custo))

W0000 00:00:1794958284.039753  140159 cuda_timer.cc:88] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


,modelo,parametros_milhoes,ms_por_imagem_em_lote,ms_por_imagem_isolada
0,resnet50,23.60,14.7,96.1
1,efficientnetb0,4.05,14.8,115.9
2,convnext_tiny,27.82,32.2,1072.8
3,densenet121,7.04,22.1,196.9
